# 01 · Comprension de los datos

> **Objetivo:** entender *Online Retail II* antes de limpiarlo. Toda decision
> de preparacion y modelado depende de este paso.

## Que vamos a hacer

1. Leer el Parquet con el **esquema declarado** (`segmentacion.data.cargar`).
2. Inspeccionar dimensiones, tipos y rango temporal.
3. Diagnosticar problemas: nulos, duplicados, cancelaciones, valores
   invalidos de `Quantity` y `Price`.
4. Dejar la Tabla 4 de la spec reproducida en este cuaderno.

## Concepto breve

El Customer Relationship Management (CRM) parte de que **no todos los
clientes son iguales**. Para segmentar necesitamos variables por cliente
construidas a partir de transacciones. El framework RFM resume el
comportamiento en tres dimensiones:

- **Recencia:** cuanto hace que compro por ultima vez.
- **Frecuencia:** cuantas veces ha comprado.
- **Monetario:** cuanto ha gastado.

El dataset es transaccional (una fila por operacion), ideal para construir
RFM en Polars.

In [1]:
# Permite importar el paquete segmentacion desde el notebook
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
import polars as pl
from segmentacion.config import RAW_DATA_DIR
from segmentacion.data.cargar import ESQUEMA, leer_parquet

PARQUET_RUTA = RAW_DATA_DIR / "online_retail_II.parquet"
assert PARQUET_RUTA.exists(), "Ejecuta `uv run python scripts/descargar_datos.py` primero."
print(f"ruta: {PARQUET_RUTA}")
print(f"ESQUEMA declarado: {ESQUEMA}")

ruta: /Users/david.palacio/Documents/data-projects-lab/projects/00-segmentacion-clientes/data/raw/online_retail_II.parquet
ESQUEMA declarado: {'Invoice': String, 'StockCode': String, 'Customer ID': Int64}


## Nulos estructurales: el nulo que **informa**

Antes de ver los conteos, un apunte conceptual que viene de la
teoria estandar de calidad de datos (Little & Rubin, 2019; Van den
Broeck et al., *The Clinical Information System DQ framework*).

No todos los nulos son errores. Hay dos tipos muy distintos:

- **Nulo faltante (missing)**: el valor **existe en la realidad**
   pero no se registro. Ejemplo clasico en salud: la temperatura del
   paciente no se midio ese dia. Imputar aqui es razonable (media,
   mediana, modelo); se asume que el valor "verdadero" existe.
- **Nulo estructural**: el campo **no aplica** al caso. Ejemplo
   historico (De Cock, 2011, *Ames Housing*): la columna
   `GarageYrBlt` esta nula para las casas que **no tienen garaje**.
   No existe un valor "verdadero"; imputar seria inventar realidad.

En Online Retail II tenemos dos nulos de distinta naturaleza:

| Columna | % de nulos | Interpretacion |
|---|---|---|
| `Description` | 0,41 % (4 382 filas) | **Estructural**: todas las filas con `Description` nula tienen `Price = 0` y sin cliente. Son **ajustes de inventario**, no ventas. |
| `Customer ID` | 22,77 % (243 007 filas) | **Mixto**: algunas pueden ser ventas sin registro del cliente (olvidar o rehusar dar el id); otras pueden ser transacciones B2C en caja fisica. **No se imputa**: nos excluyen del flujo de RFM. |

La leccion es: **antes de borrar o imputar, pregunta si el nulo
representa un hecho**. Un nulo estructural no se arregla; se
describe, y a veces se convierte en una variable binaria propia
("la factura incluye ajuste de inventario", "la venta tiene cliente
registrado").

## 1. Lectura con esquema

`leer_parquet` ya aplica la convencion de la spec (seccion 2.1): declarar el
esquema al leer evita que Polars adivine mal un `Invoice` como entero y
pierda las facturas de cancelacion (que empiezan por `C`).

In [3]:
crudo = leer_parquet(PARQUET_RUTA)
print(f"filas: {crudo.height:,}".replace(",", " "))
print(f"columnas: {crudo.width}")
crudo.schema

filas: 1 067 371
columnas: 8


Schema([('Invoice', String),
        ('StockCode', String),
        ('Description', String),
        ('Quantity', Int64),
        ('InvoiceDate', Datetime(time_unit='ms', time_zone=None)),
        ('Price', Float64),
        ('Customer ID', Int64),
        ('Country', String)])

## 2. Dimensiones y rango temporal

Confirmamos que vemos los dos periodos del dataset y los conteos basicos
que cita la Tabla 3 de la spec.

In [4]:
print(f"filas totales: {crudo.height:,}".replace(",", " "))
print(f"desde: {crudo['InvoiceDate'].min()}")
print(f"hasta: {crudo['InvoiceDate'].max()}")
print(f"paises: {crudo.select(pl.col('Country').n_unique()).item()}")
print(f"clientes distintos (incluye null): {crudo.select(pl.col('Customer ID').n_unique()).item()}")
print(f"productos distintos: {crudo.select(pl.col('StockCode').n_unique()).item()}")
print(f"facturas distintas: {crudo.select(pl.col('Invoice').n_unique()).item()}")

filas totales: 1 067 371
desde: 2009-12-01 07:45:00
hasta: 2011-12-09 12:50:00
paises: 43
clientes distintos (incluye null): 5943
productos distintos: 5305
facturas distintas: 53628


## 3. Diagnostico: nulos, duplicados, cancelaciones

Reproduce la Tabla 4 de la spec. Las cifras exactas son contrato con la spec:

| Que | Cifra |
|---|---|
| Filas sin `Customer ID` | 243 007 (22,77 %) |
| `Description` nulas | 4 382 |
| Duplicados exactos | 34 335 |
| Facturas que empiezan por `C` | 19 494 |
| `Quantity < 0` sin factura `C` | 3 457 |
| `Price = 0` / `Price < 0` | 6 202 / 5 |
| `StockCode` no estandar (no `^\d{5}`) | 6 093 |

In [5]:
n = crudo.height
sin_cliente = crudo.filter(pl.col("Customer ID").is_null()).height
descripcion_nula = crudo.filter(pl.col("Description").is_null()).height
duplicados = n - crudo.unique().height
facturas_c = crudo.filter(pl.col("Invoice").str.starts_with("C")).height
qty_neg_sin_c = crudo.filter(
    (pl.col("Quantity") < 0) & (~pl.col("Invoice").str.starts_with("C"))
).height
price_cero = crudo.filter(pl.col("Price") == 0).height
price_neg = crudo.filter(pl.col("Price") < 0).height
stock_no_std = crudo.filter(~pl.col("StockCode").str.contains(r"^\d{5}")).height

print(f"Filas sin Customer ID        : {sin_cliente:>8,} ({sin_cliente / n:.2%})".replace(",", " "))
print(f"Description nulas            : {descripcion_nula:>8,}".replace(",", " "))
print(f"Duplicados exactos           : {duplicados:>8,}".replace(",", " "))
print(f"Facturas que empiezan por C  : {facturas_c:>8,}".replace(",", " "))
print(f"Quantity < 0 sin factura C   : {qty_neg_sin_c:>8,}".replace(",", " "))
print(f"Price = 0                    : {price_cero:>8,}".replace(",", " "))
print(f"Price < 0                    : {price_neg:>8,}".replace(",", " "))
print(f"StockCode no estandar (^d5)  : {stock_no_std:>8,}".replace(",", " "))

Filas sin Customer ID        :  243 007 (22.77%)
Description nulas            :    4 382
Duplicados exactos           :   34 335
Facturas que empiezan por C  :   19 494
Quantity < 0 sin factura C   :    3 457
Price = 0                    :    6 202
Price < 0                    :        5
StockCode no estandar (^d5)  :    6 093


## 4. Hallazgos iniciales

Antes de limpiar nada, anotemos las decisiones:

| Hallazgo | Decision | Donde se aplica |
|---|---|---|
| 243 007 filas sin cliente (todas `Price = 0`, sin cliente) | Las descartamos; no sirven para RFM | `exigir_cliente` |
| 19 494 facturas que empiezan por `C` son cancelaciones | Las separamos aparte; alimentan "tasa de devolucion" en U3 | `separar_cancelaciones` |
| 6 202 filas con `Price = 0` y 5 con `Price < 0` | Las excluimos de "ventas validas" | `filtrar_validas` |
| 3 457 `Quantity < 0` sin factura `C` | Excluir: errores o devoluciones no marcadas | `filtrar_validas` |
| 34 335 duplicados exactos | Eliminamos con `unique`; se registran | `quitar_duplicados` |

> La parte accionable de este cuaderno es la columna "donde se aplica".
> Las funciones viven en `segmentacion.data.limpieza` y las llamamos
> encadenadas en el cuaderno `02_limpieza.ipynb`.

## Preguntas de reflexion

1. ¿Por que **declarar el esquema al leer** es mas robusto que castear
   despues? Piensa en que pasaria si Polars adivina `Invoice` como entero.
2. Si `StockCode` no estandar fuera un 50 % del dataset, ¿seguiria siendo
   razonable "filtrar y continuar"? ¿Cuando toca investigar antes de filtrar?
3. ¿Cuales de estos hallazgos aun representan dinero? (Pista: las
   cancelaciones.) ¿Como lo capturarias en U3?

> **Siguiente paso:** `02_limpieza.ipynb` — aplicamos las reglas y
> dejamos `data/processed/ventas_validas.parquet` listo para EDA y RFM.